This notebook allows to launch the training of the CNN-with-Attention (CNN-Attn) in order to predict demographic parameters from SNPs matrices.

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras import regularizers
from tensorflow.keras.losses import Huber
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import torch
import importlib

In [2]:
from utils import load_params, data_loading, saving_conv
from model import create_model
from config import fit_params, pred_params

## *Parameters*

In [3]:
params = load_params("params.json")

In [ ]:
print("Target:", params["target"])
print("Output directory:", params["output_dir"])

## *Data loading*

In [ ]:
X_train, Y_train_scaled, X_test, Y_test_scaled, P_train, P_test, scenarios_train, scenarios_test, scaler_Y = data_loading(params)

## *Modelling*

*First, we use convolutional part to have predictions on replicates.*

In [6]:
network_params = load_params("network_params.json")
model = create_model(params, network_params)

In [7]:
# train and pred parameters
fit_args = fit_params(X_train, P_train, Y_train_scaled)
pred_args = pred_params(X_train, X_test, P_train, P_test)

In [ ]:
# Training
history = model.fit(**fit_args)

In [ ]:
# Predictions
Y_train_pred = model.predict(pred_args["x_train"])
Y_test_pred = model.predict(pred_args["x_test"])

In [12]:
# save predictions at replicate level
saving_conv(model, history, 
            Y_train_scaled, Y_test_scaled,
            Y_train_pred, Y_test_pred,
            scenarios_train, scenarios_test,
            params["output_dir"])

*Here we apply the attention step on replicates.*

In [ ]:
from att_module import train_att_model, plot_pred_vs_obs, print_metrics

# Attention model
test_preds, test_targets = train_att_model(
    Y_train_pred, Y_train_scaled, scenarios_train,
    Y_test_pred, Y_test_scaled, scenarios_test
)

# Rescale
test_preds_scaled = scaler_Y.inverse_transform(test_preds)
test_targets_scaled = scaler_Y.inverse_transform(test_targets)

# Output dir
att_output_dir = os.path.join(params["output_dir"], "att_results")

# Save plot
plot_pred_vs_obs(test_targets_scaled, test_preds_scaled, params["target"], title="Post-Attention", output_dir=att_output_dir)

# Save metrics
print_metrics(test_targets_scaled, test_preds_scaled, params["target"], output_dir=att_output_dir)